In [1]:
import pandas as pd
import numpy as np
import json
import os

# Data folder
data_folder = "/Users/vuongdai/GitHub/bm/detrend_data/weekly/"
# Data file
data_file = "ts_weekly_values.csv"
# Time file
data_file_time = "ts_weekly_datetimes.csv"
# Anomaly info file
anomaly_info_file = "anomaly_info.json"

# Read time file (skip first row as header, each column = one time series)
df_time = pd.read_csv(data_folder + data_file_time, skiprows=1, delimiter=",", header=None)

anomaly_info = {}

for col in df_time.columns:
    # Extract non-NaN time values for this column
    times = df_time[col].dropna().values

    if len(times) == 0:
        print(f"Column {col}: no valid timestamps, skipping.")
        continue

    # Randomly pick 10 times (or fewer if series has less than 10)
    n_pick = min(10, len(times))
    picked_indices = np.sort(np.random.choice(len(times), size=n_pick, replace=False))
    picked_times = times[picked_indices]

    # Format times as strings (strip time component if present, keep date only)
    picked_times_str = []
    for t in picked_times:
        try:
            picked_times_str.append(pd.Timestamp(t).strftime("%Y-%m-%d"))
        except Exception:
            picked_times_str.append(str(t))

    # Randomly generate 10 level and trend magnitudes between 1e-3 and 0.5
    level_magnitudes = np.random.uniform(1e-3, 0.5, size=n_pick).tolist()
    trend_magnitudes = np.random.uniform(1e-3, 0.5, size=n_pick).tolist()

    # Round for readability
    level_magnitudes = [round(v, 3) for v in level_magnitudes]
    trend_magnitudes = [round(v, 3) for v in trend_magnitudes]

    anomaly_info[str(col)] = {
        "anomaly_time": picked_times_str,
        "anomaly_magnitude": {
            "level": level_magnitudes,
            "trend": trend_magnitudes
        }
    }

# Save to JSON
output_path = os.path.join(data_folder, anomaly_info_file)
with open(output_path, "w") as f:
    json.dump(anomaly_info, f, indent=4)

print(f"anomaly_info.json saved to: {output_path}")
print(f"Total columns processed: {len(anomaly_info)}")

# Preview first entry
first_key = next(iter(anomaly_info))
print(f"\nPreview of column '{first_key}':")
print(json.dumps(anomaly_info[first_key], indent=4))

anomaly_info.json saved to: /Users/vuongdai/GitHub/bm/detrend_data/weekly/anomaly_info.json
Total columns processed: 101

Preview of column '0':
{
    "anomaly_time": [
        "2012-11-11",
        "2015-01-25",
        "2015-06-28",
        "2015-07-05",
        "2016-09-11",
        "2017-02-19",
        "2017-07-16",
        "2017-09-10",
        "2021-07-18",
        "2022-11-20"
    ],
    "anomaly_magnitude": {
        "level": [
            0.428,
            0.301,
            0.116,
            0.159,
            0.434,
            0.015,
            0.348,
            0.316,
            0.087,
            0.399
        ],
        "trend": [
            0.233,
            0.209,
            0.484,
            0.43,
            0.205,
            0.16,
            0.027,
            0.118,
            0.304,
            0.173
        ]
    }
}


In [2]:
# GenerateAnomaly(data=df, time=df_time, anomaly_info=anomaly_info)